# Autonomous AI SecOps Log Analyzer: An Agentic Framework for Threat Profiling

**Group 84** — Yash Nimbalkar, Darshan Nikrad

This notebook reproduces the full experiment: synthetic log generation → statistical Z-score triage → agentic AI threat classification → evaluation against a rule-based baseline. It runs entirely **offline** (deterministic classifier) so no API key is required. To use the LLM analyst instead, set `OPENAI_API_KEY` and change the provider to `openai`.

## Setup

In [ ]:
import os, sys, json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, ROOT)
os.chdir(ROOT)

from secops.config import load_settings
from secops.data_generator import generate_logs, summarize
from secops.stats_engine import detect_anomalies
from secops.rule_engine import apply_rules, scan_ratio_flag
from secops.pipeline import run_pipeline

settings = load_settings('config.yaml')
settings.data['llm']['provider'] = 'offline'   # zero-cost demo
print('config loaded:', settings.seed, 'minutes', settings.minutes)

## Phase 1 — Synthetic data with ground-truth labels

In [ ]:
df, ground_truth = generate_logs(
    seed=settings.seed, minutes=settings.minutes,
    base_rate_per_min=settings.base_rate_per_min,
    background_ips=settings.background_ips, out_dir='data')
print(json.dumps(summarize(df, ground_truth), indent=2))

## Phase 2 — Statistical baseline (Z-score over behavioural features)

In [ ]:
anomalies, feature_frame = detect_anomalies(
    df, window_minutes=settings.window_minutes,
    z_threshold=settings.z_threshold, robust=settings.robust_stats)
print(f'{len(anomalies)} anomalous (IP, window) pairs flagged from {len(feature_frame)} ip-windows')
top = feature_frame.nlargest(6, 'anomaly_score')
top[['src_ip', 'window', 'anomaly_score', 'top_feature', 'n_requests']]

## Phase 3 & 4 — Agentic triage + end-to-end pipeline

In [ ]:
results = run_pipeline(settings)
agent = results['agent']
print(f"provider={agent['provider']} | candidates={len(agent['candidate_ips'])} "
      f"| reports={len(agent['reports'])} | cost=${agent['llm_cost_usd']}")

## Phase 5 — Evaluation: hybrid agentic vs rule-based baseline

In [ ]:
from secops.evaluation import compare_detectors
comp = pd.DataFrame(results['evaluation']['detectors'])
comp[['detector', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1',
      'false_positive_rate', 'alerts_issued', 'detection_latency_mean_min']]

## Figures for the paper

In [ ]:
%run notebooks/make_figures.py
from IPython.display import Image, display
for name in ['fig1_traffic_timeline', 'fig2_anomaly_scores', 'fig4_detection_quality']:
    display(Image(f'outputs/figures/{name}.png'))

## Conclusion

The statistical stage provides **high recall** (catches every attacker IP) but suffers **alert fatigue** (many false positives). The agentic layer reasons over the full feature vector and **eliminates false positives while recovering attacks that single-threshold rules miss** (e.g. moderate floods). The hybrid pipeline achieves the best precision–recall balance in every experiment — the central claim of the paper.